# Mini projet sur speech recognition par whisper

1.   installation des independences
1.   Importer les bibliothèques nécessaires
2.   téléchargement les fichiers nécessaires
2.   Visualisation
2.   Application Audio par Whisper
2.   Exemple de video youtube par Whisper
2.   Analyse et résultat de sentiment Whisper et NLTK(vader)



# installation des independences

In [98]:
!pip install -U ffmpeg openai-whisper
!pip install pytube -q

ERROR: Operation cancelled by user


# Importer librairies.

In [96]:
# Importation du modèle Whisper pour les tâches liées à la parole
import whisper

# Importation du module YouTube de la bibliothèque pytube pour travailler avec les vidéos YouTube
from pytube import YouTube

# Importation du kit de traitement du langage naturel (nltk) pour les tâches de traitement du langage naturel
import nltk

# Importation des modules stopwords, word_tokenize et sent_tokenize de nltk.corpus et nltk.tokenize
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

# Importation de la bibliothèque pandas pour la manipulation et l'analyse des données
import pandas as pd

# Importation de la bibliothèque numpy pour les opérations numériques
import numpy as np

# Importation de la bibliothèque de visualisation matplotlib.pyplot pour les graphiques
import matplotlib.pyplot as plt

# Importation de la bibliothèque scipy pour le traitement du signal
from scipy import signal

# Importation des modules wavfile et display d'IPython pour le traitement audio et l'affichage
from scipy.io import wavfile
import IPython.display as ipd

# téléchargement les fichiers nécessaires

In [ ]:
# Téléchargement de la liste des mots vides (stopwords) de NLTK
nltk.download("stopwords")

# Téléchargement des modèles de tokenisation de phrases et de mots de NLTK
nltk.download("punkt")

# Téléchargement du lexique VADER pour l'analyse des sentiments de NLTK
nltk.download('vader_lexicon')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

# Visualisation

In [ ]:
from IPython.display import Audio, display
audio_path = "/content/harvard.wav"
display(Audio(audio_path, autoplay=True))

In [8]:
def data_reader(audio_path, window_size=80, step_size=40, eps=1e-10):
    # Lire le fichier audio
    sample_rate, samples = wavfile.read(audio_path)

    # Vérifier si l'audio a plusieurs canaux
    if len(samples.shape) > 1:
        # Si plusieurs canaux, calculer le spectrogramme pour chaque canal
        spec_list = []
        for channel in range(samples.shape[1]):
            freqs, times, spec = signal.spectrogram(
                samples[:, channel], fs=sample_rate, window='hann', detrend=False
            )
            # Ajouter le spectrogramme log-transformé à la liste
            spec_list.append(np.log(spec.T.astype(np.float32) + eps))

        # Empiler les spectrogrammes pour chaque canal le long d'un nouvel axe
        spec_array = np.stack(spec_list, axis=-1)

        # Retourner les valeurs de fréquence, les valeurs temporelles,
        # le tableau de spectrogrammes log-transformés, le taux d'échantillonnage et les échantillons bruts
        return freqs, times, spec_array, sample_rate, samples

    else:
        # Si un seul canal, calculer le spectrogramme comme précédemment
        freqs, times, spec = signal.spectrogram(
            samples, fs=sample_rate, window='hann', detrend=False
        )

        # Retourner les valeurs de fréquence, les valeurs temporelles, le spectrogramme log-transformé,
        # le taux d'échantillonnage et les échantillons bruts
        return freqs, times, np.log(spec.T.astype(np.float32) + eps), sample_rate, samples

In [ ]:
# Chemin du fichier audio

# Appel de la fonction data_reader pour obtenir les données audio
freqs, times, spectrogram, sample_rate, samples = data_reader(
    audio_path, window_size=20, step_size=10, eps=1e-10
)

# Nombre de canaux dans l'audio
num_channels = samples.shape[1] if len(samples.shape) > 1 else 1

# Ajuster la largeur et la hauteur du graphique en modifiant le paramètre figsize
fig, axs = plt.subplots(num_channels, 2, figsize=(16, 5 * num_channels))
fig.suptitle(f"Waveform and Spectrogram of {audio_path}", fontsize=16)

for channel in range(num_channels):
    # Tracer le waveform brut pour chaque canal
    axs[channel, 0].set_title(f'Raw wave - Channel {channel + 1}')
    axs[channel, 0].set_ylabel('Amplitude')
    axs[channel, 0].plot(np.linspace(0, len(samples) / sample_rate, len(samples)), samples[:, channel])

    # Tracer le spectrogramme pour chaque canal
    axs[channel, 1].set_title(f'Spectrogram - Channel {channel + 1}')
    axs[channel, 1].imshow(
        spectrogram[:, :, channel].T,
        aspect='auto',
        origin='lower',
        extent=[times.min(), times.max(), freqs.min(), freqs.max()]
    )
    axs[channel, 1].set_yticks(freqs[::16])
    axs[channel, 1].set_xticks(times[::16])
    axs[channel, 1].set_ylabel('Freqs in Hz')
    axs[channel, 1].set_xlabel('Seconds')

# Afficher les graphiques
plt.show()

# Application Audio par  Whisper

In [77]:
def transcribe(audio, model_size="base"):
    """
    Transcrit un fichier audio en utilisant le modèle Whisper.

    Args:
    - audio (str): Chemin vers le fichier audio.
    - model_size (str): Taille du modèle à utiliser ("tiny", "base", "small", "medium").

    Returns:
    - str: Texte transcrit.
    """
    # Vérifier si le type de modèle fourni est "tiny", "base", "small", ou "medium"
    if model_size not in ["tiny", "base", "small", "medium"]:
        raise ValueError("Invalid model_size. Please use 'tiny', 'base', 'small', or 'medium'.")

    # Charger le modèle Whisper en fonction de la taille spécifiée
    model = whisper.load_model(model_size)

    # Effectuer la transcription et obtenir le résultat
    result = model.transcribe(audio, verbose=True)

    # Retourner le texte transcrit
    return result["text"]


In [10]:
content=transcribe("/content/harvard.wav","base")
content

100%|███████████████████████████████████████| 139M/139M [00:04<00:00, 31.2MiB/s]
/usr/local/lib/python3.10/dist-packages/whisper/transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: English
[00:00.000 --> 00:04.480]  The stale smell of old beer lingers.
[00:04.480 --> 00:07.020]  It takes heat to bring out the odor.
[00:07.020 --> 00:09.940]  A cold dip restores health and zest.
[00:09.940 --> 00:12.620]  A salt pickle tastes fine with ham.
[00:12.620 --> 00:15.080]  Tacos al pastor are my favorite.
[00:15.080 --> 00:17.600]  A zestful food is the hot cross bun.


' The stale smell of old beer lingers. It takes heat to bring out the odor. A cold dip restores health and zest. A salt pickle tastes fine with ham. Tacos al pastor are my favorite. A zestful food is the hot cross bun.'

# Exemple de video youtube par Whisper et spacy(vader)

In [78]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from string import punctuation
from heapq import nlargest

def summarize(text, per):
    nlp = spacy.load('en_core_web_sm')
    doc= nlp(text)
    tokens=[token.text for token in doc]
    word_frequencies={}
    for word in doc:
        if word.text.lower() not in list(STOP_WORDS):
            if word.text.lower() not in punctuation:
                if word.text not in word_frequencies.keys():
                    word_frequencies[word.text] = 1
                else:
                    word_frequencies[word.text] += 1
    max_frequency=max(word_frequencies.values())
    for word in word_frequencies.keys():
        word_frequencies[word]=word_frequencies[word]/max_frequency
    sentence_tokens= [sent for sent in doc.sents]
    sentence_scores = {}
    for sent in sentence_tokens:
        for word in sent:
            if word.text.lower() in word_frequencies.keys():
                if sent not in sentence_scores.keys():
                    sentence_scores[sent]=word_frequencies[word.text.lower()]
                else:
                    sentence_scores[sent]+=word_frequencies[word.text.lower()]
    select_length=int(len(sentence_tokens)*per)
    summary=nlargest(select_length, sentence_scores,key=sentence_scores.get)
    final_summary=[word.text for word in summary]
    summary=''.join(final_summary)
    return summary

In [79]:
def youtube_resume(youtube_link, summarize_percent=0.1):
        # Charger la vidéo YouTube
        youtube_video_content = YouTube(youtube_link)

        # Sélectionner le flux audio
        audio_stream = youtube_video_content.streams.filter(only_audio=True).first()

        # Télécharger l'audio
        audio_path = "/content/video2.mp4"
        audio_stream.download("/content/", filename="video2.mp4")

        #Transcrire l'audio avec Whisper
        model = whisper.load_model("base")
        result = model.transcribe(audio_path)
        print("La langue détectée est :", result["language"])

        # # Créer un DataFrame Pandas avec le résumé et le texte complet
        data = {'Summary': [summarize(result['text'], summarize_percent)], 'Full Text': [result["text"]]}
        df = pd.DataFrame(data)

        return df

In [81]:
a=youtube_resume("https://www.youtube.com/watch?v=KnHbejJKQVk")
print(a)

/usr/local/lib/python3.10/dist-packages/whisper/transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


La langue détectée est : en
                                             Summary  \
0  For a moment to breathe and taste life without...   

                                           Full Text  
0   I am so tired of being strong. I'm tired of b...  


# analyse et résultat de sentiment Whisper et NLTK

In [49]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

def text_sentiment(x):
    sent=vader.polarity_scores(x)
    max_key = max(sent, key=sent.get)
    if max_key == 'neg':
          return 'negative'
    elif max_key == 'neu':
          return 'neutral'
    elif max_key == 'pos':
          return 'positive'
    elif max_key == 'compound':
          return 'composed'
    else:
          return 'unknown'

In [50]:
def sentiment(audio, model_type="base", nb_segments_to_process=6, polarity_threshold=0.1):
    try:
        # Charger le modèle Whisper en fonction du type spécifié
        if model_type not in ["tiny", "base", "small", "medium"]:
            raise ValueError("Invalid model_type. Please use 'tiny', 'base', 'small', or 'medium'.")
        model = whisper.load_model(model_type)

        # Transcrire l'audio
        result = model.transcribe(audio, verbose=False)

        # Créer un DataFrame à partir des segments transcrits
        earnings_call_df = pd.DataFrame.from_dict(result['segments'])

        # Filtrer la colonne texte et retirer les colonnes restantes
        earnings_call_df = earnings_call_df[['text']]

        # Sélectionner un nombre spécifié de segments aléatoires depuis le DataFrame
        earnings_call_df = earnings_call_df.sample(n=nb_segments_to_process)

        # Appliquer l'analyse de sentiment à la colonne 'text'
        earnings_call_df['sentiment'] = earnings_call_df['text'].apply(text_sentiment)

        # Classer les sentiments en fonction du seuil de polarité
        earnings_call_df['sentiment'] = np.where(
            earnings_call_df['sentiment'] == 'composed', 'neutral',
            np.where(earnings_call_df['sentiment'] == 'positive', 'positive', 'negative')
        )

        return earnings_call_df

    except Exception as e:
        print(f"Une erreur s'est produite : {str(e)}")
        return None

In [95]:
sentiment("/content/video2.mp4")

/usr/local/lib/python3.10/dist-packages/whisper/transcribe.py:115: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Detected language: English


100%|██████████| 7871/7871 [00:19<00:00, 403.18frames/s]


,text,sentiment
10,looks like without dark clouds.,negative
1,I'm tired of being praised for bearing the we...,negative
0,I am so tired of being strong.,negative
13,It's bizarre to think I only appreciate the s...,negative
3,"Instead of what it is, heavy.",negative
5,I am exhausted.,negative
